# 지도학습 분류 — 당뇨병 위험군 탐지 재현 실험

정확도만 높이는 것이 아니라 실제 당뇨 환자를 놓치는 FN을 줄이는 것을 목표로 한다. 주 지표는 Recall과 F2이며 PR-AUC, Precision, Specificity도 함께 보고한다.

- 실행 기준 시각: `20260814_062642`
- 원본 노트북은 `01_원본보관`에 수정 없이 보관했다.
- 이 노트북은 최종 재현 파이프라인이다. 과거의 모든 탐색 실험과 출력은 원본 보관본에서 확인할 수 있다.
- 모델 선택은 검증 세트에서만 수행하고, 테스트 세트는 최종 보고에 사용한다.
- 모든 비교는 동일 분할과 동일 평가지표를 사용하며 학습·추론 시간도 기록한다.

## 1. 환경·데이터 로드

In [1]:
from pathlib import Path
from time import perf_counter
import json, warnings
import numpy as np
import pandas as pd
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             fbeta_score, roc_auc_score, average_precision_score,
                             confusion_matrix)
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
SEED = 42
ROOT = Path.cwd()
ARTIFACTS = ROOT / "03_실험결과"
ARTIFACTS.mkdir(exist_ok=True)

data_path = Path(kagglehub.dataset_download("alexteboul/diabetes-health-indicators-dataset"))
csv_path = data_path / "diabetes_binary_health_indicators_BRFSS2015.csv"
df = pd.read_csv(csv_path)
print("data:", csv_path.name)
print("shape:", df.shape, "positive_rate:", round(df['Diabetes_binary'].mean(), 4))
print("duplicates:", int(df.duplicated().sum()), "missing:", int(df.isna().sum().sum()))

data: diabetes_binary_health_indicators_BRFSS2015.csv
shape: (253680, 22) positive_rate: 0.1393
duplicates: 24206 missing: 0


## 2. 검증 설계

원본의 한계를 보완했다.

1. 전체 데이터를 먼저 훈련 60% / 검증 20% / 테스트 20%로 계층 분할한다.
2. 임계값은 검증 세트 F2가 최대가 되도록 선택한다. 테스트 세트로 임계값을 고르지 않는다.
3. 중복 제거 실험은 훈련 세트의 다수 클래스에만 적용한다. 검증·테스트 행은 삭제하지 않는다.
4. BMI·정신건강일·신체건강일은 유효 범위가 정의된 변수이므로 IQR만으로 환자를 삭제하지 않는다.

In [2]:
X = df.drop(columns="Diabetes_binary").copy()
y = df["Diabetes_binary"].astype(int).copy()
X["Risk_Sum"] = X[["HighBP", "HighChol", "HeartDiseaseorAttack"]].sum(axis=1)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)
print({"train": X_train.shape, "validation": X_val.shape, "test": X_test.shape})
print({"train_positive": y_train.mean(), "validation_positive": y_val.mean(), "test_positive": y_test.mean()})

def choose_threshold(y_true, prob, beta=2.0):
    thresholds = np.linspace(0.05, 0.80, 151)
    scores = [fbeta_score(y_true, prob >= t, beta=beta, zero_division=0) for t in thresholds]
    i = int(np.argmax(scores))
    return float(thresholds[i]), float(scores[i])

def classification_metrics(y_true, prob, threshold):
    pred = (np.asarray(prob) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "specificity": tn / (tn + fp),
        "f1": f1_score(y_true, pred, zero_division=0),
        "f2": fbeta_score(y_true, pred, beta=2, zero_division=0),
        "roc_auc": roc_auc_score(y_true, prob),
        "pr_auc": average_precision_score(y_true, prob),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

{'train': (152208, 22), 'validation': (50736, 22), 'test': (50736, 22)}
{'train_positive': np.float64(0.1393356459581625), 'validation_positive': np.float64(0.13932907600126143), 'test_positive': np.float64(0.13932907600126143)}


## 3. 비교 실험: 기본 RF, 클래스 가중치, 다양한 모델 앙상블, 다수 클래스 중복 축소

In [3]:
models = {}
fit_seconds = {}

def timed_fit(name, model, X_fit, y_fit):
    start = perf_counter()
    model.fit(X_fit, y_fit)
    fit_seconds[name] = perf_counter() - start
    models[name] = model
    print(name, "fit_seconds=", round(fit_seconds[name], 3))

timed_fit(
    "RF_baseline",
    RandomForestClassifier(n_estimators=160, min_samples_leaf=2, random_state=SEED, n_jobs=-1),
    X_train, y_train,
)
timed_fit(
    "RF_balanced",
    RandomForestClassifier(n_estimators=160, min_samples_leaf=2, class_weight="balanced",
                           random_state=SEED, n_jobs=-1),
    X_train, y_train,
)
timed_fit(
    "Logistic_balanced",
    Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000,
                                     random_state=SEED, n_jobs=-1)),
    ]),
    X_train, y_train,
)
timed_fit(
    "LGBM_balanced",
    LGBMClassifier(n_estimators=350, learning_rate=0.04, num_leaves=31,
                   subsample=0.85, colsample_bytree=0.85, class_weight="balanced",
                   random_state=SEED, n_jobs=-1, verbosity=-1),
    X_train, y_train,
)

train_joined = X_train.copy()
train_joined["__target__"] = y_train.values
majority = train_joined[train_joined["__target__"] == 0].drop_duplicates()
minority = train_joined[train_joined["__target__"] == 1]
train_dedup = pd.concat([majority, minority]).sample(frac=1, random_state=SEED)
X_train_dedup = train_dedup.drop(columns="__target__")
y_train_dedup = train_dedup["__target__"]
print("majority duplicate reduction:", len(X_train), "->", len(train_dedup))
timed_fit(
    "LGBM_majority_dedup",
    LGBMClassifier(n_estimators=350, learning_rate=0.04, num_leaves=31,
                   subsample=0.85, colsample_bytree=0.85, class_weight="balanced",
                   random_state=SEED, n_jobs=-1, verbosity=-1),
    X_train_dedup, y_train_dedup,
)

RF_baseline fit_seconds= 1.921


RF_balanced fit_seconds= 1.757


Logistic_balanced fit_seconds= 2.83


LGBM_balanced fit_seconds= 0.975
majority duplicate reduction: 152208 -> 140964


LGBM_majority_dedup fit_seconds= 0.932


In [4]:
prob_val = {}
prob_test = {}
infer_seconds = {}
for name, model in models.items():
    prob_val[name] = model.predict_proba(X_val)[:, 1]
    start = perf_counter()
    prob_test[name] = model.predict_proba(X_test)[:, 1]
    infer_seconds[name] = perf_counter() - start

# 강사 조언의 '다양한 앙상블'을 반영: 선형 Logistic + 비선형 LightGBM 확률 평균
prob_val["Diverse_ensemble"] = (prob_val["Logistic_balanced"] + prob_val["LGBM_balanced"]) / 2
prob_test["Diverse_ensemble"] = (prob_test["Logistic_balanced"] + prob_test["LGBM_balanced"]) / 2
fit_seconds["Diverse_ensemble"] = fit_seconds["Logistic_balanced"] + fit_seconds["LGBM_balanced"]
infer_seconds["Diverse_ensemble"] = infer_seconds["Logistic_balanced"] + infer_seconds["LGBM_balanced"]

rows = []
thresholds = {}
for name in prob_val:
    threshold = 0.5 if name == "RF_baseline" else choose_threshold(y_val, prob_val[name], beta=2)[0]
    thresholds[name] = threshold
    val_m = classification_metrics(y_val, prob_val[name], threshold)
    test_m = classification_metrics(y_test, prob_test[name], threshold)
    row = {"model": name, "fit_seconds": fit_seconds[name],
           "inference_ms_per_1000": infer_seconds[name] * 1_000_000 / len(X_test)}
    row.update({f"val_{k}": v for k, v in val_m.items()})
    row.update({f"test_{k}": v for k, v in test_m.items()})
    rows.append(row)

results = pd.DataFrame(rows).sort_values("val_f2", ascending=False).reset_index(drop=True)
selected_name = results.loc[0, "model"]
print("selected_on_validation:", selected_name)
display(results[["model", "val_f2", "test_precision", "test_recall", "test_f1", "test_f2",
                 "test_roc_auc", "test_pr_auc", "fit_seconds", "inference_ms_per_1000"]])
results.to_csv(ARTIFACTS / "classification_experiment_results.csv", index=False)

predictions = pd.DataFrame({
    "actual": y_test.values,
    "baseline_probability": prob_test["RF_baseline"],
    "selected_probability": prob_test[selected_name],
    "selected_prediction": (prob_test[selected_name] >= thresholds[selected_name]).astype(int),
})
predictions.to_csv(ARTIFACTS / "classification_test_predictions.csv", index=False)

selected_on_validation: Diverse_ensemble


,model,val_f2,test_precision,test_recall,test_f1,test_f2,test_roc_auc,test_pr_auc,fit_seconds,inference_ms_per_1000
0,Diverse_ensemble,0.609538,0.281412,0.842128,0.421855,0.602165,0.826546,0.424064,3.805547,1.575824
1,LGBM_balanced,0.607612,0.290513,0.822181,0.429326,0.601881,0.827072,0.426687,0.975214,1.384551
2,LGBM_majority_dedup,0.607402,0.286324,0.834913,0.426414,0.603612,0.826995,0.426404,0.931703,1.593912
3,Logistic_balanced,0.603634,0.276267,0.842128,0.416046,0.597403,0.820653,0.399691,2.830333,0.191272
4,RF_balanced,0.596895,0.277577,0.826567,0.415591,0.592284,0.812871,0.396700,1.757362,2.130503
5,RF_baseline,0.182584,0.564320,0.150799,0.238000,0.176695,0.815682,0.409436,1.920874,2.131295


## 4. 성능 향상 근거: 테스트 부트스트랩 95% 신뢰구간

In [5]:
rng = np.random.default_rng(SEED)
base_pred = (prob_test["RF_baseline"] >= 0.5).astype(int)
sel_pred = (prob_test[selected_name] >= thresholds[selected_name]).astype(int)
y_arr = y_test.to_numpy()
diffs = []
for _ in range(1000):
    idx = rng.integers(0, len(y_arr), len(y_arr))
    diffs.append(fbeta_score(y_arr[idx], sel_pred[idx], beta=2, zero_division=0) -
                 fbeta_score(y_arr[idx], base_pred[idx], beta=2, zero_division=0))
ci = np.quantile(diffs, [0.025, 0.975])
summary = {
    "selected_model": selected_name,
    "selected_threshold": thresholds[selected_name],
    "test_f2_improvement_vs_rf_baseline": float(np.mean(diffs)),
    "bootstrap_95pct_ci": [float(ci[0]), float(ci[1])],
    "selection_rule": "maximum validation F2",
    "test_rows": int(len(y_test)),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
(ARTIFACTS / "classification_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

{
  "selected_model": "Diverse_ensemble",
  "selected_threshold": 0.425,
  "test_f2_improvement_vs_rf_baseline": 0.42514415907470243,
  "bootstrap_95pct_ci": [
    0.41372825403778835,
    0.43583298511259
  ],
  "selection_rule": "maximum validation F2",
  "test_rows": 50736
}


278

## 5. 해석 원칙

- Recall 증가는 FN 감소라는 업무 효과로 해석한다.
- 임계값을 낮추면 Recall은 대체로 증가하지만 FP도 증가한다. 따라서 Precision·Specificity를 함께 제시한다.
- 부트스트랩 신뢰구간이 0을 포함하면, 단일 점수 차이를 확정적 개선이라고 표현하지 않는다.
- 이 데이터는 관찰형 설문 데이터이므로 예측 근거를 인과관계로 해석하지 않는다.